**STEAM MACHINE LEARNING**

**Integrantes**:

João Pedro Costa Cruz

João Vitor Souza Tavares

Lucas Antônio Araújo Santos

**Link para o dataset original**:

https://www.kaggle.com/datasets/fronkongames/steam-games-dataset

**Descrição do problema**:

O dataset escolhido traz informações sobre jogos virtuais da plataforma Steam. A ideia é determinar se um jogo é um "Sucesso". Um jogo é considerado um sucesso se ele tiver uma taxa de aprovação de reviews positivos maior ou igual a 80% e ao menos 100 reviews.

O problema foi estruturado como uma tarefa de **Classificação** porque o nosso objetivo final é mapear os dados de entrada para uma decisão categórica e discreta (Sucesso vs. Não Sucesso). Isso permite que tomadores de decisão (como publicadoras de jogos) façam avaliações qualitativas e utilizem métricas de diagnóstico de erro cruciais (como a taxa de falsos positivos e falsos negativos através de matrizes de confusão) para minimizar os riscos de investimento.

In [ ]:
#==================================
#   IMPORTAÇÃO DAS BIBLIOTECAS
#==================================

# Manipulação dos dados
import pandas as pd
import numpy as np

# Visualização gráfica
import matplotlib.pyplot as plt
import seaborn as sns

# Configurações do ambiente de plotagem
%matplotlib inline
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Ignorar avisos de atualizações de bibliotecas
import warnings
warnings.filterwarnings('ignore')

print("Configurações do ambiente concluídas.")

In [ ]:
#============================================
#   DATASET E DEFINIÇÃO DO ATRIBUTO-ALVO
#============================================

# URL do dataset
url_dataset = "https://raw.githubusercontent.com/LucasAAraujoS/steam-machine-learning/refs/heads/main/steam_games_clean.csv"

# Carrega o arquivo usando o ponto e vírgula (;) como delimitador
df = pd.read_csv(url_dataset, sep=';')

#----------------------------------------------------------------------------------------------------------------------------

# Cálculo do volume total de avaliações (reviews)
df['Total_Reviews'] = df['Positive'] + df['Negative']

# Cálculo da taxa de aprovação de um jogo
df['Approval_Rate'] = np.where(df['Total_Reviews'] > 0, df['Positive'] / df['Total_Reviews'], 0.0)

# Definição do atributo-alvo
df['Success'] = ((df['Approval_Rate'] >= 0.80) & (df['Total_Reviews'] >= 100)).astype(int)

print(f"Dataset carregado com sucesso!")
print(f"Dimensões do DataFrame: {df.shape[0]} linhas e {df.shape[1]} colunas.")

## 1. Engenharia do Alvo e Estatística Descritiva

Esta seção cobre: distribuição do atributo-alvo (`Success`), estatística descritiva geral do dataset e remoção das colunas usadas para construir o alvo (evitando data leakage).

In [ ]:
#==================================
#   DISTRIBUIÇÃO DO ATRIBUTO-ALVO
#==================================

# Calculo da frequ~encia dos dados
success_counts = df['Success'].value_counts().sort_index()
success_pct = df['Success'].value_counts(normalize=True).sort_index() * 100

print("Contagem por classe:")
print(success_counts)
print("\nProporção por classe (%):")
print(success_pct.round(2))
print("\n")

# Plotagem do gráfico
fig, ax = plt.subplots()
sns.countplot(x='Success', data=df, palette=['#c0392b', '#27ae60'], ax=ax) # Desenha o gráfico
ax.set_xticklabels(['Não Sucesso (0)', 'Sucesso (1)']) # Altera os nomes de "0" e "1 para o que está dentro dos colchetes
ax.set_title('Distribuição da variável alvo: Sucesso vs. Não Sucesso')
ax.set_xlabel('')
ax.set_ylabel('Quantidade de jogos')
for i, v in enumerate(success_counts):
    ax.text(i, v + 1500, f'{v:,}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

**Interpretação:**

O dataset apresenta um forte desbalanceamento de classes: **89,56% dos jogos (112.718) são classificados como "Não Sucesso"**, enquanto apenas **10,44% (13.137) são "Sucesso"**, ou seja, uma proporção aproximada de 9 para 1.

Isso reflete uma realidade do mercado da Steam: a grande maioria dos jogos publicados na plataforma não atinge um patamar consistente de avaliações positivas somado a um volume relevante de reviews — o que é coerente com o fato de a Steam ser uma plataforma aberta, com milhares de lançamentos pequenos, indie ou pouco divulgados.

Esse desbalanceamento é uma informação crítica para as próximas etapas do projeto: ele justifica (a) o uso de `stratify=y` na separação treino/teste para preservar essa proporção em ambos os conjuntos, e (b) a priorização do F1-Score e da matriz de confusão em vez da acurácia simples na avaliação dos modelos, já que um classificador ingênuo que sempre previsse "Não Sucesso" já acertaria quase 90% dos casos sem aprender nada de útil.

### Estatística descritiva geral do dataset

In [ ]:
#==================================
#   ESTATÍSTICA DESCRITIVA GERAL
#==================================

print(f"O dataset possui {df.shape[0]:,} registros e {df.shape[1]} atributos.\n")

print("Tipos de dados por coluna:")
print(df.dtypes)

valores_ausentes = df.isnull().sum()
print("\nValores ausentes por coluna (apenas colunas com ausências):")
print(valores_ausentes[valores_ausentes > 0])

print(f"\nLinhas totalmente duplicadas: {df.duplicated().sum()}")
print(f"Nomes de jogos duplicados (podem ser DLCs/remasters/reentradas): {df['Name'].duplicated().sum()}")

df[['Price', 'Positive', 'Negative', 'Achievements']].describe()

**Interpretação:**

O dataset contém 125.855 jogos e 14 atributos, com um volume ausente concentrado em variáveis textuais/categóricas: `Developers` (8.449), `Publishers` (8.922), `Categories` (8.966) e `Genres` (8.423) — em torno de 6,7% das linhas não têm gênero ou desenvolvedor informado, o que provavelmente reflete jogos cadastrados de forma incompleta na Steam (ex: *playtests* ou títulos removidos). Há também 18 linhas totalmente duplicadas e 1.210 nomes repetidos, que precisarão ser investigados na Etapa 2 (podem ser duplicatas reais ou apenas jogos com o mesmo nome).

Quanto ao preço, `a média (US$ 4,81) é bem maior que a mediana (US$ 2,39)`, o que indica uma distribuição assimétrica à direita: a maioria dos jogos é barata, mas alguns títulos caros (até US$ 999,98) puxam a média para cima. O mesmo padrão, ainda mais extremo, aparece em `Positive` e `Negative`: a mediana de reviews positivas é de apenas 4, mas a média passa de 1.000, evidenciando a presença de outliers fortes — alguns poucos jogos com milhões de reviews (blockbusters) distorcem completamente a média em relação à realidade da maioria dos títulos da plataforma.

### Remoção das colunas usadas para construir o alvo (evitando *data leakage*)

In [ ]:
#===================================================
#   REMOÇÃO DAS COLUNAS DE VAZAMENTO (DATA LEAKAGE)
#===================================================

# Positive, Negative, Total_Reviews e Approval_Rate foram usadas para CALCULAR
# o alvo (Success). Mantê-las como atributos preditivos permitiria ao modelo
# 'colar' a resposta em vez de aprender padrões reais.

# Mantemos 'df' completo para a análise exploratória e criamos
# 'df_model' -- sem essas colunas -- para ser o ponto de partida do
# que vem a seguir (pré-processamento e modelagem).

leakage_cols = ['Positive', 'Negative', 'Total_Reviews', 'Approval_Rate']

df_model = df.drop(columns=leakage_cols)

print("Colunas removidas para evitar data leakage:", leakage_cols)
print(f"\nColunas restantes para modelagem ({df_model.shape[1]}):")
print(list(df_model.columns))

**Interpretação:**

`Positive`, `Negative`, `Total_Reviews` e `Approval_Rate` não podem seguir para o treinamento porque foram exatamente as variáveis usadas para definir `Success`: um modelo treinado com elas não estaria aprendendo a prever sucesso a partir de características do jogo (preço, gênero, plataforma etc.), e sim reconstruindo uma fórmula que já conhecemos — isso é chamado de vazamento de dados (*data leakage*) e infla artificialmente as métricas sem gerar um modelo útil na prática.

`df_model` guarda a versão do dataset já livre desse problema, pronta para ser usada. `df` (com todas as colunas originais) continua disponível para as análises de correlação e distribuição que vêm a seguir.

# 2. Correlação e Relações com o Alvo.

Esta seção cobre: correlação das variáveis numéricas com `Success`, gráfico de dispersão preço x taxa de aprovação, e tabelas de frequência cruzando plataformas e gêneros com a taxa de sucesso.

### Correlação das variáveis numéricas com o alvo

In [ ]:
#==================================
#   CORRELAÇÃO COM O ALVO
#==================================

# Extrai o ano de lançamento a partir da data (string -> datetime)
df['Release_Year'] = pd.to_datetime(df['Release date'], errors='coerce').dt.year

# Faz a correlação retirando a linha 'sucess', pois a correlação sempre seria 1
num_cols = ['Price', 'Achievements', 'Release_Year']
correlacoes = df[num_cols + ['Success']].corr()['Success'].drop('Success')
print("Correlação (Pearson) com Success:")
print(correlacoes.sort_values(ascending=False))

# Reliza a plotagem do gráfico
fig, ax = plt.subplots()
correlacoes.sort_values().plot(kind='barh', color='#2980b9', ax=ax)
ax.set_title('Correlação das variáveis numéricas com Success')
ax.set_xlabel('Coeficiente de correlação')
ax.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

**Interpretação:**

Nenhuma variável numérica isolada tem correlação linear forte com `Success` — o que é esperado, já que sucesso de um jogo é multifatorial. `Price` (0,036) e `Achievements` (0,047) praticamente não têm relação linear direta com o sucesso, sugerindo que preço e quantidade de conquistas, por si só, não determinam a recepção do jogo.

O destaque é `Release_Year`, com correlação negativa de -0,23: jogos mais recentes tendem a ter taxa de sucesso menor. Isso provavelmente não significa que jogos novos são piores, e sim um efeito do próprio critério do alvo (`Total_Reviews >= 100`) — jogos lançados há mais tempo tiveram mais tempo para acumular reviews e ultrapassar esse limiar, enquanto lançamentos recentes ainda estão a caminho. Vale mencionar essa limitação na discussão crítica.

### Gráfico de dispersão: Preço x Taxa de Aprovação

In [ ]:
#==================================
#   DISPERSÃO: PREÇO x APROVAÇÃO
#==================================

fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(
    data=df[df['Total_Reviews'] >= 100],  # foca em jogos com volume relevante de reviews
    x='Price', y='Approval_Rate', hue='Success',
    palette={0: '#c0392b', 1: '#27ae60'}, alpha=0.4, ax=ax
)
ax.set_title('Preço x Taxa de Aprovação (jogos com \u2265 100 reviews)')
ax.set_xlabel('Preço (US$)')
ax.set_ylabel('Taxa de aprovação')
ax.set_xlim(0, 100)  # a maioria dos jogos custa bem menos que isso. Corta outliers extremos de preço
ax.axhline(0.80, color='black', linestyle='--', linewidth=0.8, label='Limiar de Sucesso (80%)')
ax.legend(title='Success')
plt.tight_layout()
plt.show()

**Interpretação:**

O gráfico mostra que jogos de sucesso (verde) aparecem espalhados por praticamente toda a faixa de preço, sem se concentrarem nem nos jogos mais baratos nem nos mais caros — reforçando a baixa correlação encontrada acima. O que fica visível é que jogos com preço próximo de zero têm uma dispersão de aprovação muito maior (do 0% ao 100%), enquanto jogos pagos tendem a se agrupar um pouco mais para cima na taxa de aprovação, possivelmente porque cobrar um preço filtra parte do público e reduz avaliações de "experimentação" impulsivas.

### Tabela de frequência: Plataformas x Taxa de Sucesso

In [ ]:
#==================================
#   PLATAFORMAS x TAXA DE SUCESSO
#==================================

plataformas = ['Windows', 'Mac', 'Linux']
dados_grafico = {}

for plataforma in plataformas:
    taxa = df.groupby(plataforma)['Success'].mean() * 100
    dados_grafico[plataforma] = taxa

# Transforma os resultados em um DataFrame fácil de plotar
df_plataformas = pd.DataFrame(dados_grafico)
df_plataformas.index = ['Não Compatível (0)', 'Compatível (1)']

print("Tabela de taxas de sucesso (%):")
print(df_plataformas.round(2))

# Realiza a plotagem do gráfico de barras agrupadas
fig, ax = plt.subplots(figsize=(8, 5))
df_plataformas.T.plot(kind='bar', color=['#e74c3c', '#2ecc71'], ax=ax, edgecolor='black', alpha=0.8)

ax.set_title('Taxa de Sucesso por Compatibilidade de Sistema Operacional')
ax.set_ylabel('Taxa de Sucesso (%)')
ax.set_xlabel('Plataformas')
ax.set_xticklabels(plataformas, rotation=0) # Mantém os nomes Windows/Mac/Linux retos
ax.grid(axis='y', linestyle='--', alpha=0.5) # Linhas de grade ao fundo para ajudar a ler as alturas
ax.legend(title='Suporte')

plt.tight_layout()
plt.show()

**Interpretação:**

Jogos compatíveis com Windows têm taxa de sucesso de 10,44%, contra apenas 2,27% dos que não são compatíveis — mas isso provavelmente reflete que jogos sem suporte a Windows são um grupo muito pequeno e atípico na Steam (plataforma majoritariamente Windows), não necessariamente uma relação causal.

Mais interessante é o padrão em Mac e Linux: jogos compatíveis com Mac têm taxa de sucesso de 19,88% (contra 8,47% sem suporte), e com Linux, 17,88% (contra 9,35%). Isso sugere que oferecer suporte multiplataforma pode estar associado a jogos mais bem cuidados tecnicamente ou voltados a um público mais engajado, embora não devamos interpretar isso como "adicionar suporte a Mac/Linux causa sucesso" sem mais investigação.

### Tabela de frequência: Gêneros x Taxa de Sucesso

In [ ]:
#==================================
#   GÊNEROS x TAXA DE SUCESSO
#==================================

# 'Genres' é multi-valorado (um jogo pode ter vários gêneros separados por vírgula),
# então precisamos 'explodir' a coluna antes de agrupar.
generos = df[['Genres', 'Success']].dropna(subset=['Genres']).copy()
generos['Genres'] = generos['Genres'].str.split(',') # Transforma os gêneros em uma lista
generos = generos.explode('Genres') # Duplica a linha 
generos['Genres'] = generos['Genres'].str.strip() # Remove espaços em branco

# Filtra os 10 gêneros mais populares
top_generos = generos['Genres'].value_counts().head(10).index
tabela_generos = (
    generos[generos['Genres'].isin(top_generos)]
    .groupby('Genres')['Success']
    .agg(taxa_sucesso='mean', quantidade='count')
    .sort_values('taxa_sucesso', ascending=False)
)
tabela_generos['taxa_sucesso'] = (tabela_generos['taxa_sucesso'] * 100).round(2)
print(tabela_generos)

# Plotagem do gráfico
fig, ax = plt.subplots()
tabela_generos['taxa_sucesso'].sort_values().plot(kind='barh', color='#8e44ad', ax=ax)
ax.set_title('Taxa de sucesso por gênero (10 gêneros mais frequentes)')
ax.set_xlabel('Taxa de sucesso (%)')
plt.tight_layout()
plt.show()

**Interpretação:**

Entre os 10 gêneros mais frequentes, **RPG** (13,46%), **Adventure** (12,66%) e **Simulation** (11,73%) têm as maiores taxas de sucesso, enquanto **Early Access** (6,16%) e **Free To Play** (8,25%) têm as menores. Faz sentido: jogos em Early Access ainda estão em desenvolvimento e tendem a acumular críticas sobre bugs e conteúdo incompleto, e jogos Free To Play atraem um público muito maior e mais heterogêneo, o que aumenta a chance de reviews negativas de usuários que não são o público-alvo real do jogo. Já **Indie**, apesar de ser o gênero mais frequente do dataset (82.884 jogos), tem taxa de sucesso próxima da mediana (10,84%), mostrando que ser indie não é, por si só, indicativo de melhor ou pior recepção.

# 3. Distribuição das Variáveis Preditivas

Esta seção cobre: histogramas e boxplots das variáveis numéricas preditivas, identificação de outliers, e verificação de inconsistências no dataset.

### Histograma do Preço (Price)

In [ ]:
#==================================
#   DISTRIBUIÇÃO DO PREÇO
#==================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df['Price'], bins=50, ax=axes[0], color='#2980b9')
axes[0].set_title('Distribuição do Preço (escala normal)')
axes[0].set_xlabel('Preço (US$)')

sns.histplot(df[df['Price'] > 0]['Price'], bins=50, log_scale=True, ax=axes[1], color='#2980b9')
axes[1].set_title('Distribuição do Preço > 0 (escala log)')
axes[1].set_xlabel('Preço (US$, escala log)')

plt.tight_layout()
plt.show()

print(f"Jogos com preço igual a zero (gratuitos): {(df['Price']==0).sum():,} ({(df['Price']==0).mean()*100:.2f}%)")

**Interpretação:**

Na escala normal, a distribuição do preço é extremamente concentrada perto de zero, com uma cauda longa — a visualização em escala log (à direita) é necessária para enxergar a forma real da distribuição entre os jogos pagos, que se aproxima de uma distribuição log-normal, comum em preços de mercado (muitos produtos baratos, poucos caros).

21,18% dos jogos (26.661) são totalmente gratuitos (Price = 0), o que é coerente com o modelo Free To Play amplamente adotado na Steam e reforça por que essa variável sozinha tem baixo poder preditivo isolado (visto nas analises anteriores): jogos gratuitos e pagos aparecem em ambas as classes de sucesso.

### Boxplot do Preço por classe do alvo

In [ ]:
#==================================
#   BOXPLOT: PREÇO x SUCCESS
#==================================

fig, ax = plt.subplots(figsize=(8, 6))
sns.boxplot(data=df, x='Success', y='Price', ax=ax, showfliers=False)
ax.set_xticklabels(['Não Sucesso (0)', 'Sucesso (1)'])
ax.set_title('Preço por classe (outliers extremos ocultados para leitura)')
ax.set_xlabel('')
ax.set_ylabel('Preço (US$)')
plt.tight_layout()
plt.show()

print(df.groupby('Success')['Price'].describe()[['mean','50%','75%','max']])

**Interpretação:**

Jogos de sucesso têm mediana de preço mais alta `(US$ 3,99) do que os de não sucesso (US$ 1,99)`, e o terceiro quartil também é maior `(US$ 8,74 vs. US$ 4,99)`. Isso não contradiz a baixa correlação linear vista nas analises anteriores — a relação parece existir, mas não é linear: jogos de sucesso tendem a se concentrar numa faixa de preço um pouco mais alta, sem que "preço maior" garanta sucesso de forma proporcional.

### Distribuição de Achievements e identificação de outliers (IQR)

In [ ]:
#==================================
#   DISTRIBUIÇÃO DE ACHIEVEMENTS E OUTLIERS (REGRA DO IQR)
#==================================

fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(df[df['Achievements'] <= 200]['Achievements'], bins=50, color='#e67e22', ax=ax)
ax.set_title('Distribuição de Achievements (até 200, para visualização)')
ax.set_xlabel('Quantidade de conquistas')
plt.tight_layout()
plt.show()

# Identificação de outliers pela regra do IQR (1.5x)
q1, q3 = df['Achievements'].quantile([0.25, 0.75])
iqr = q3 - q1
limite_superior = q3 + 1.5 * iqr # Fórmula padrão da estatística

n_outliers = (df['Achievements'] > limite_superior).sum()
print(f"Q1={q1}, Q3={q3}, IQR={iqr}")
print(f"Limite superior (Q3 + 1.5*IQR): {limite_superior}")
print(f"Jogos acima do limite (outliers): {n_outliers:,} ({n_outliers/len(df)*100:.2f}%)")
print(f"Máximo de conquistas no dataset: {df['Achievements'].max():,}")
print(f"Jogos com 0 conquistas: {(df['Achievements']==0).sum():,} ({(df['Achievements']==0).mean()*100:.2f}%)")

**Interpretação:**

Quase metade dos jogos (48,57%) não tem nenhuma conquista cadastrada, e a regra do IQR (Intervalo Interquartil) aponta 8.606 jogos (6,84%) como outliers — jogos com mais de ~48 conquistas, chegando a um máximo de 9.821. Como essa variável é fortemente concentrada em zero com uma cauda muito longa, ela também deve ser tratada com cautela (ex: escalonamento robusto a outliers, como `RobustScaler`, em vez de `StandardScaler`).

### Verificação de inconsistências no dataset

In [ ]:
#==================================
#   VERIFICAÇÃO DE INCONSISTÊNCIAS
#==================================

print("Valores de Price negativos:", (df['Price'] < 0).sum())
print("Valores de Achievements negativos:", (df['Achievements'] < 0).sum())

# 'Estimated owners' vem como faixa em texto (ex: '0 - 20000'); não é numérica direto
print("\nExemplos de valores em 'Estimated owners':")
print(df['Estimated owners'].value_counts().head(5))

# Jogos com muitas reviews mas 0 owners estimados (faixa '0 - 0') seriam uma inconsistência
inconsistentes = df[(df['Estimated owners'] == '0 - 0') & (df['Total_Reviews'] > 0)]
print(f"\nJogos na faixa '0 - 0' de owners mas com reviews > 0 (possível inconsistência): {len(inconsistentes):,}")

**Interpretação:**

Não há valores negativos em `Price` ou `Achievements`, e nenhum jogo classificado na faixa "0 - 0" de proprietários estimados possui reviews registradas — bons sinais de integridade básica dos dados, sem inconsistências lógicas entre essas variáveis. Já `Estimated owners` está armazenada como faixas de texto (ex: `"0 - 20000"`), não como número — isso não é um erro, mas uma característica da fonte de dados, e precisa ser tratada (ex: convertida para o ponto médio da faixa) caso seja usada como atributo preditivo posteriormente.